# Exercise 2: Extending the toy grammar

*Finite-State Morphology · version 0.1.0*

> Add the accusative *-(y)I* to `code/ch06-toy-fst/`: a new archiphoneme for the four-way vowel, a buffer *y* after vowel-final stems (*arabayı*), and the *kitap/kitab-* alternation before vowels (*kitabı*). Add accepted and rejected test forms for each change, and run the tests.

**Setup.** `pip install "turkicnlp[hfst]==0.5.0"`. These notebooks compile lexc and twolc grammars, which needs the compiled `hfst` package; on PyPI it had wheels for Python 3.11 but not for 3.13 when checked on 19 October 2026, so use Python 3.11 or 3.12 if `pip` cannot find it. The Apertium transducers are downloaded once with `turkicnlp.download`.

The extension adds three archiphonemes to the toy grammar of the chapter:

- `{I}` for the four-way vowel (*ı, i, u, ü*), with one rule per value, each looking back over consonants to the nearest vowel;
- `{y}` for the buffer, which surfaces as *y* after a vowel and as nothing otherwise;
- `{P}` as the last letter of *kitap*, which surfaces as *b* before a vowel and as *p* elsewhere.

Writing the alternation as a property of the stem (`kita{P}`) follows the chapter: it is lexically conditioned, so a stem such as *top* 'ball' would simply keep a plain *p*. The stem *kol* 'arm' is added so that the rounded back value *u* is tested. The left contexts of the harmony rules allow `{y}:0`, so that a buffer that is not realised does not block harmony.

In [ ]:
import tempfile
from pathlib import Path
import hfst
import turkicnlp
from turkicnlp.resources.registry import ModelRegistry

def compile_grammar(lexc_text, twolc_text=None):
    """Compile a lexc lexicon (and optional twolc rules); return (analyser, generator) in lookup format."""
    work = Path(tempfile.mkdtemp())
    (work / "g.lexc").write_text(lexc_text, encoding="utf-8")
    gen = hfst.compile_lexc_file(str(work / "g.lexc"))
    if twolc_text:
        (work / "g.twolc").write_text(twolc_text, encoding="utf-8")
        hfst.compile_twolc_file(str(work / "g.twolc"), str(work / "g.twol.hfst"))
        stream, rules = hfst.HfstInputStream(str(work / "g.twol.hfst")), []
        while not stream.is_eof():
            rules.append(stream.read())
        stream.close()
        gen.compose_intersect(tuple(rules))
    gen.minimize()
    ana = hfst.HfstTransducer(gen)
    ana.invert()
    ana.minimize()
    for t in (ana, gen):
        t.convert(hfst.ImplementationType.HFST_OLW_TYPE)
    return ana, gen

def apertium(lang, script, kind="automorf"):
    """An Apertium transducer distributed with TurkicNLP (downloaded on first use)."""
    turkicnlp.download(lang, processors=["morph"])
    path = ModelRegistry.get_model_path(lang, "morph", "apertium", script=script) / f"{lang}.{kind}.hfst"
    stream = hfst.HfstInputStream(str(path))
    fst = stream.read()
    stream.close()
    return fst

lookup = lambda fst, s: sorted({out for out, _ in fst.lookup(s)})

In [ ]:
LEXC = r"""! Toy grammar extended with the accusative -(y)I, the buffer y and the kitap/kitab- alternation.
! {A} = a/e;  {I} = ı/i/u/ü;  {D} = d/t;  {y} = buffer y or nothing;  {P} = p/b
Multichar_Symbols
%<n%> %<pl%> %<loc%> %<abl%> %<acc%> %{A%} %{I%} %{D%} %{y%} %{P%}

LEXICON Root
Nouns ;

LEXICON Nouns
ev%<n%>:ev               Number ;   ! house
araba%<n%>:araba         Number ;   ! car
göz%<n%>:göz             Number ;   ! eye
kol%<n%>:kol             Number ;   ! arm (new: back rounded vowel)
kitap%<n%>:kita%{P%}     Number ;   ! book: final p alternates with b

LEXICON Number
%<pl%>:l%{A%}r     Case ;
                   Case ;

LEXICON Case
                       # ;
%<loc%>:%{D%}%{A%}     # ;
%<abl%>:%{D%}%{A%}n    # ;
%<acc%>:%{y%}%{I%}     # ;
"""

TWOLC = r"""Alphabet
a b c ç d e f g ğ h ı i j k l m n o ö p r s ş t u ü v y z
%{A%}:a %{A%}:e %{I%}:ı %{I%}:i %{I%}:u %{I%}:ü %{D%}:d %{D%}:t %{y%}:y %{y%}:0 %{P%}:p %{P%}:b ;

Sets
Vowel = a e ı i o ö u ü ;
FrontV = e i ö ü ;
BackUnr = a ı ;
FrontUnr = e i ;
BackRnd = o u ;
FrontRnd = ö ü ;
Cons = b c ç d f g ğ h j k l m n p r s ş t v y z ;
Voiceless = ç f h k p s ş t ;

Rules

"{A} is e after a front vowel, otherwise a"
%{A%}:e <=> :FrontV [ :Cons | %{y%}:0 ]* _ ;

"{I} is ı after a or ı"
%{I%}:ı <=> :BackUnr [ :Cons | %{y%}:0 ]* _ ;

"{I} is i after e or i"
%{I%}:i <=> :FrontUnr [ :Cons | %{y%}:0 ]* _ ;

"{I} is u after o or u"
%{I%}:u <=> :BackRnd [ :Cons | %{y%}:0 ]* _ ;

"{I} is ü after ö or ü"
%{I%}:ü <=> :FrontRnd [ :Cons | %{y%}:0 ]* _ ;

"{y} is y after a vowel, otherwise nothing"
%{y%}:y <=> :Vowel _ ;

"{D} is t after a voiceless consonant"
%{D%}:t <=> :Voiceless _ ;

"{P} is b before a vowel"
%{P%}:b <=> _ [ %{y%}:0 ]* :Vowel ;
"""
analyser, generator = compile_grammar(LEXC, TWOLC)

In [ ]:
accept = {"evi": "ev<n><acc>", "arabayı": "araba<n><acc>", "gözü": "göz<n><acc>", "kolu": "kol<n><acc>",
          "kitabı": "kitap<n><acc>", "evleri": "ev<n><pl><acc>", "arabaları": "araba<n><pl><acc>",
          "kitapları": "kitap<n><pl><acc>", "kitapta": "kitap<n><loc>", "kitap": "kitap<n>",
          "kollardan": "kol<n><pl><abl>", "evlerde": "ev<n><pl><loc>"}
reject = ["evı", "arabaı", "arabayi", "gözi", "kolı", "kitapı", "kitabda", "kitab", "kitaba"]
failures = [w for w, gold in accept.items() if lookup(analyser, w) != [gold]]
failures += [w for w in reject if lookup(analyser, w)]
print("failures:", failures or "none")

TAGS = ["<n>", "<n><acc>", "<n><loc>", "<n><abl>", "<n><pl>", "<n><pl><acc>", "<n><pl><loc>", "<n><pl><abl>"]
for stem in ["ev", "araba", "göz", "kol", "kitap"]:
    forms = [lookup(generator, stem + t) for t in TAGS]
    assert all(len(f) == 1 and lookup(analyser, f[0]) == [stem + t] for f, t in zip(forms, TAGS))
    print(f"{stem:6}", " ".join(f[0] for f in forms))
print("all 40 cells generate one form and analyse back to their tags")

## Worked solution

Output of `experiments/ch06_exercises/run_ch06_exercises.py` on 19 October 2026 (TurkicNLP 0.5.0, HFST 3.16.0.1, Python 3.11):

```
failures: none
ev     ev evi evde evden evler evleri evlerde evlerden
araba  araba arabayı arabada arabadan arabalar arabaları arabalarda arabalardan
göz    göz gözü gözde gözden gözler gözleri gözlerde gözlerden
kol    kol kolu kolda koldan kollar kolları kollarda kollardan
kitap  kitap kitabı kitapta kitaptan kitaplar kitapları kitaplarda kitaplardan
all 40 cells generate one form and analyse back to their tags
```

All twelve forms that must be accepted get exactly the expected analysis, all nine that must be rejected get none, and the 40 paradigm cells (five stems × eight cells) generate one form each and analyse back to their tags. What each test checks:

- **Four-way vowel `{I}`:** *evi*, *gözü*, *kolu*, *arabayı* are accepted; *evı*, *gözi*, *kolı* are rejected.
- **Buffer `{y}`:** *arabayı* is accepted, *arabaı* is rejected (no buffer) and *arabayi* is rejected (buffer present, harmony wrong).
- **Alternation `{P}`:** *kitabı* is accepted, *kitapı* and the bare *kitab* are rejected; before a consonant *p* stays (*kitapta*, *kitapları*), and *kitabda* is rejected.
- *kitaba* is rejected only because the toy grammar has no dative yet (Your turn, 2).

One point needs care in twolc: when the buffer is not realised, the pair `{y}:0` stands between the stem and the suffix vowel, so the left contexts of the harmony rules and the right context of the `{P}` rule have to allow it explicitly.

## Your turn — extension

1. Add *top* 'ball' (no alternation: *topu*) and *çiçek* 'flower' (*çiçeği*, with *k* → *ğ*). Which archiphoneme does *çiçek* need?
2. Add the dative *-(y)A* and check *arabaya*, *kitaba*, *kola*.
3. Save the two files next to `tr-toy.lexc` and extend `build_and_test.py` with these tests.